
# 10 — Product Pulse Decision Engine

**Final analytical notebook**

This notebook combines Product Pulse outputs into a standardized decision layer for a future API/dashboard.

It does not train another large model.


In [ ]:

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.home() / "Desktop" / "resume_projects" / "ProductPulse"
PROCESSED_02 = PROJECT_ROOT / "data" / "processed" / "02_cleaned"
PROCESSED_07 = PROJECT_ROOT / "data" / "processed" / "07_modeling"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

PROCESSED_07.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("PROJECT_ROOT:", PROJECT_ROOT)


## 1. Load earlier compact results

In [ ]:

def result_csv(folder, filename):
    path = PROJECT_ROOT / "results" / folder / "tables" / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run the required earlier notebook first."
        )
    return pd.read_csv(path)

funnel_metrics = result_csv(
    "03_retailrocket_behavior_analytics",
    "funnel_metrics.csv",
)
customer_summary = result_csv(
    "04_online_retail_customer_product_analytics",
    "customer_summary.csv",
)
experiment_conclusion = result_csv(
    "05_cookie_cats_experiment_analysis",
    "experiment_conclusion.csv",
)
model_comparison = result_csv(
    "08_behavior_and_customer_prediction",
    "model_comparison.csv",
)
rr_top_prospects = result_csv(
    "08_behavior_and_customer_prediction",
    "retailrocket_top_prospects.csv",
)
or_top_customers = result_csv(
    "08_behavior_and_customer_prediction",
    "online_retail_top_customers.csv",
)
uplift_model_comparison = result_csv(
    "09_uplift_modeling_and_targeting",
    "uplift_model_comparison.csv",
)
top_uplift_targets = result_csv(
    "09_uplift_modeling_and_targeting",
    "top_uplift_targets.csv",
)

print("Loaded all required prior results.")


## 2. Funnel recommendation

In [ ]:

funnel_lookup = dict(zip(funnel_metrics["metric"], funnel_metrics["value"]))

viewer_to_cart = float(funnel_lookup.get("viewer_to_cart_pct", np.nan))
cart_to_tx = float(funnel_lookup.get("cart_to_transaction_pct", np.nan))

if pd.notna(viewer_to_cart) and pd.notna(cart_to_tx) and viewer_to_cart < cart_to_tx:
    funnel_action = "Prioritize view-to-cart conversion improvements"
    funnel_reason = (
        f"Viewer→cart is {viewer_to_cart:.2f}% versus "
        f"cart→transaction at {cart_to_tx:.2f}%."
    )
else:
    funnel_action = "Prioritize cart-to-transaction conversion improvements"
    funnel_reason = (
        f"Viewer→cart is {viewer_to_cart:.2f}% and "
        f"cart→transaction is {cart_to_tx:.2f}%."
    )

funnel_decision = pd.DataFrame([{
    "capability": "Behavior Analytics",
    "entity_type": "funnel",
    "entity_id": "retailrocket",
    "decision_type": "funnel_optimization",
    "score": min(viewer_to_cart, cart_to_tx),
    "recommended_action": funnel_action,
    "confidence": "descriptive",
    "reason": funnel_reason,
    "source_notebook": "03",
}])

display(funnel_decision)


## 3. Experiment recommendation

In [ ]:

row7 = experiment_conclusion[
    experiment_conclusion["metric"] == "retention_7"
]

if len(row7):
    r = row7.iloc[0]
    diff = float(r["absolute_difference_gate40_minus_gate30"])
    p_value = float(r["p_value"])

    if p_value < 0.05 and diff < 0:
        experiment_action = "KEEP gate_30"
        experiment_reason = (
            f"gate_40 reduced day-7 retention by "
            f"{abs(diff) * 100:.2f} percentage points "
            f"(p={p_value:.4g})."
        )
        experiment_confidence = "high"
    elif p_value < 0.05 and diff > 0:
        experiment_action = "ADOPT gate_40"
        experiment_reason = (
            f"gate_40 improved day-7 retention by "
            f"{diff * 100:.2f} percentage points "
            f"(p={p_value:.4g})."
        )
        experiment_confidence = "high"
    else:
        experiment_action = "NO CHANGE"
        experiment_reason = (
            f"No statistically significant day-7 retention difference "
            f"(p={p_value:.4g})."
        )
        experiment_confidence = "moderate"
else:
    experiment_action = "REVIEW EXPERIMENT"
    experiment_reason = "Day-7 retention result unavailable."
    experiment_confidence = "low"

experiment_decision = pd.DataFrame([{
    "capability": "Experimentation",
    "entity_type": "experiment",
    "entity_id": "cookie_cats_gate_test",
    "decision_type": "product_experiment",
    "score": float(row7.iloc[0]["absolute_difference_gate40_minus_gate30"]) if len(row7) else np.nan,
    "recommended_action": experiment_action,
    "confidence": experiment_confidence,
    "reason": experiment_reason,
    "source_notebook": "05",
}])

display(experiment_decision)


## 4. Predictive targeting decisions

In [ ]:

best_predictive = (
    model_comparison[model_comparison["split"] == "test"]
    .sort_values(["task", "pr_auc"], ascending=[True, False])
    .groupby("task", as_index=False)
    .first()
)

predictive_registry = best_predictive[
    ["task", "model", "roc_auc", "pr_auc", "f1", "precision_at_10pct"]
].copy()
predictive_registry["decision"] = "Use for ranked targeting / prioritization"

rr25 = rr_top_prospects.head(25)
rr_decisions = pd.DataFrame({
    "capability": "Behavior Prediction",
    "entity_type": "visitor",
    "entity_id": rr25["visitorid"].astype(str),
    "decision_type": "conversion_propensity",
    "score": rr25["purchase_probability"],
    "recommended_action": "Prioritize for conversion intervention",
    "confidence": "model_score",
    "reason": "Top-ranked probability of future transaction.",
    "source_notebook": "08",
})

or25 = or_top_customers.head(25)
or_decisions = pd.DataFrame({
    "capability": "Customer Prediction",
    "entity_type": "customer",
    "entity_id": or25["CustomerID"].astype(str),
    "decision_type": "repeat_purchase_propensity",
    "score": or25["repeat_purchase_probability"],
    "recommended_action": "Prioritize for retention / repeat-purchase campaign",
    "confidence": "model_score",
    "reason": "Top-ranked probability of purchase in the next 60 days.",
    "source_notebook": "08",
})

display(predictive_registry)


## 5. Uplift targeting decisions

In [ ]:

best_uplift = uplift_model_comparison.iloc[0]

uplift_strategy = pd.DataFrame([{
    "capability": "Uplift Modeling",
    "entity_type": "campaign",
    "entity_id": "criteo_conversion_treatment",
    "decision_type": "treatment_targeting",
    "score": best_uplift["normalized_qini_area"],
    "recommended_action": "Target the highest predicted-uplift segment first",
    "confidence": "model_validation",
    "reason": (
        f"Best model: {best_uplift['model']}; "
        f"uplift@10%={best_uplift['uplift_at_10pct']:.6f}, "
        f"uplift@20%={best_uplift['uplift_at_20pct']:.6f}."
    ),
    "source_notebook": "09",
}])

u25 = top_uplift_targets.head(25)
uplift_target_decisions = pd.DataFrame({
    "capability": "Uplift Targeting",
    "entity_type": "ranked_candidate",
    "entity_id": u25["target_rank"].astype(str),
    "decision_type": "individual_treatment_effect",
    "score": u25["predicted_uplift"],
    "recommended_action": u25["recommended_action"],
    "confidence": "model_score",
    "reason": "Ranked by estimated incremental conversion effect.",
    "source_notebook": "09",
})

display(uplift_strategy)


## 6. Unified decision table

In [ ]:

decision_table = pd.concat(
    [
        funnel_decision,
        experiment_decision,
        uplift_strategy,
        rr_decisions,
        or_decisions,
        uplift_target_decisions,
    ],
    ignore_index=True,
)

display(decision_table.head(20))
print("Decision records:", len(decision_table))


## 7. Executive summary and model registry

In [ ]:

def metric_value(table, name):
    rows = table.loc[table["metric"] == name, "value"]
    return rows.iloc[0] if len(rows) else np.nan

repeat_pct = float(metric_value(customer_summary, "repeat_buyer_pct"))

executive_summary = pd.DataFrame([
    {
        "area": "Funnel",
        "headline": funnel_action,
        "evidence": funnel_reason,
    },
    {
        "area": "Customer",
        "headline": "Use repeat-purchase propensity for retention prioritization",
        "evidence": f"Historical repeat-buyer rate: {repeat_pct:.2f}%.",
    },
    {
        "area": "Experiment",
        "headline": experiment_action,
        "evidence": experiment_reason,
    },
    {
        "area": "Treatment Targeting",
        "headline": "Use individualized uplift instead of blanket treatment",
        "evidence": uplift_strategy.iloc[0]["reason"],
    },
])

model_registry = predictive_registry.rename(
    columns={
        "task": "capability",
        "model": "selected_model",
    }
)

model_registry = pd.concat([
    model_registry,
    pd.DataFrame([{
        "capability": "Criteo individual treatment effect",
        "selected_model": best_uplift["model"],
        "roc_auc": np.nan,
        "pr_auc": np.nan,
        "f1": np.nan,
        "precision_at_10pct": np.nan,
        "decision": "Use for uplift-ranked treatment targeting",
    }]),
], ignore_index=True)

display(executive_summary)
display(model_registry)


In [ ]:

fig_capabilities, ax = plt.subplots(figsize=(9, 4.8))
counts = decision_table["capability"].value_counts().sort_values()
ax.barh(counts.index, counts.values)
ax.set_title("Product Pulse — Decision Outputs by Capability")
ax.set_xlabel("Decision records")
fig_capabilities.tight_layout()
plt.show()


## 8. Save final analytical outputs

In [ ]:

RESULTS_DIR = PROJECT_ROOT / "results" / "10_product_pulse_decision_engine"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RESULT_TABLES = {
    "executive_summary": executive_summary,
    "decision_table": decision_table,
    "model_registry": model_registry,
    "predictive_registry": predictive_registry,
    "funnel_decision": funnel_decision,
    "experiment_decision": experiment_decision,
    "uplift_strategy": uplift_strategy,
}
RESULT_FIGURES = {
    "decision_outputs_by_capability": fig_capabilities,
}

for name, table in RESULT_TABLES.items():
    table.to_csv(TABLES_DIR / f"{name}.csv", index=False)
for name, fig in RESULT_FIGURES.items():
    fig.savefig(FIGURES_DIR / f"{name}.png", dpi=200, bbox_inches="tight")

decision_table.to_json(
    RESULTS_DIR / "decision_table.json",
    orient="records",
    indent=2,
)

print("=" * 68)
print("PRODUCT PULSE ANALYTICAL PIPELINE COMPLETE")
print("=" * 68)
print("Tables saved :", len(RESULT_TABLES))
print("Figures saved:", len(RESULT_FIGURES))
print("Decision JSON saved.")
print("Results:", RESULTS_DIR)
